# ISL-Translator: INCLUDE 50GB Dataset → Keypoints (Fully Self-Contained)


In [1]:
# Cell 1: Install EXACT pinned versions that work on Kaggle Python 3.12
!pip install -q "mediapipe==0.10.14" "opencv-python-headless" "tqdm" "numpy<2.2" "protobuf>=4.25"
print("✅ Dependencies installed.")

   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/35.7 MB 111.9 MB/s eta 0:00:01

   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/35.7 MB 134.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━ 20.5/35.7 MB 161.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━ 26.0/35.7 MB 157.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 31.5/35.7 MB 157.4 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 35.7/35.7 MB 162.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/294.9 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 14.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.


✅ Dependencies installed.


In [2]:
# Cell 2: Inline KeypointExtractor (no external imports needed)
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path
from tqdm.auto import tqdm
import requests
import zipfile
import shutil
import json
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ISL-Extractor")

# Verify mediapipe.solutions exists
assert hasattr(mp, 'solutions'), f"mediapipe {mp.__version__} missing .solutions! Need 0.10.14"
print(f"✅ mediapipe {mp.__version__} loaded with .solutions API")

class KeypointExtractor:
    NUM_POSE = 33
    NUM_FACE = 468
    NUM_HAND = 21
    TOTAL = 543  # 33 + 468 + 21 + 21

    def __init__(self):
        self.holistic = mp.solutions.holistic.Holistic(
            static_image_mode=False,
            model_complexity=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        print(f"✅ MediaPipe Holistic initialized ({self.TOTAL} landmarks/frame)")

    def extract_frame(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.holistic.process(rgb)
        kp = np.zeros((self.TOTAL, 3), dtype=np.float32)
        idx = 0
        if results.pose_landmarks:
            for lm in results.pose_landmarks.landmark:
                kp[idx] = [lm.x, lm.y, lm.visibility]; idx += 1
        else:
            idx += self.NUM_POSE
        if results.face_landmarks:
            for lm in results.face_landmarks.landmark:
                kp[idx] = [lm.x, lm.y, lm.z]; idx += 1
        else:
            idx += self.NUM_FACE
        if results.left_hand_landmarks:
            for lm in results.left_hand_landmarks.landmark:
                kp[idx] = [lm.x, lm.y, lm.z]; idx += 1
        else:
            idx += self.NUM_HAND
        if results.right_hand_landmarks:
            for lm in results.right_hand_landmarks.landmark:
                kp[idx] = [lm.x, lm.y, lm.z]; idx += 1
        return kp

    def extract_video(self, video_path):
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return None
        all_kp = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            all_kp.append(self.extract_frame(frame))
        cap.release()
        if len(all_kp) == 0:
            return None
        return np.array(all_kp, dtype=np.float32)

    def close(self):
        self.holistic.close()

print("✅ KeypointExtractor class defined.")

2026-03-09 11:36:47.473664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773056207.727647      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773056207.800957      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773056208.403136      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773056208.403196      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773056208.403199      17 computation_placer.cc:177] computation placer alr

✅ mediapipe 0.10.14 loaded with .solutions API
✅ KeypointExtractor class defined.


In [3]:
# Cell 3: Download, Extract, Process loop
TEMP_DIR = Path('/kaggle/working/temp')
OUTPUT_DIR = Path('/kaggle/working/data/processed/include')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, target_path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(target_path, 'wb') as f, tqdm(desc=target_path.name, total=total, unit='iB', unit_scale=True) as bar:
        for chunk in r.iter_content(1024*1024):
            f.write(chunk)
            bar.update(len(chunk))

# Fetch file list from Zenodo
print("Fetching INCLUDE metadata from Zenodo...")
meta = requests.get("https://zenodo.org/api/records/4010759").json()
zip_files = [(f['key'], f['links']['self']) for f in meta.get('files', []) if f['key'].endswith('.zip')]
print(f"Found {len(zip_files)} zip files to process.")

# Initialize extractor ONCE (reuse across all zips)
extractor = KeypointExtractor()
total_extracted = 0
total_failed = 0

for zip_idx, (zip_name, zip_url) in enumerate(zip_files):
    print(f"\n{'='*60}")
    print(f"[{zip_idx+1}/{len(zip_files)}] Processing {zip_name}")
    print(f"{'='*60}")
    
    # Clean temp directory
    if TEMP_DIR.exists():
        shutil.rmtree(TEMP_DIR)
    TEMP_DIR.mkdir(parents=True)
    
    try:
        # Download
        zip_path = TEMP_DIR / zip_name
        download_file(zip_url, zip_path)
        
        # Extract zip
        print("Extracting zip...")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(TEMP_DIR)
        zip_path.unlink()  # Delete zip to save space
        
        # Find ALL video files recursively (case insensitive)
        video_exts = ('.mp4', '.avi', '.mov', '.mkv', '.webm')
        all_videos = [f for f in TEMP_DIR.rglob('*') if f.is_file() and f.suffix.lower() in video_exts]
        print(f"Found {len(all_videos)} video files.")
        
        if len(all_videos) == 0:
            print(f"⚠️ No videos in {zip_name}, skipping.")
            continue
        
        # Extract keypoints from each video
        for vid_path in tqdm(all_videos, desc="Extracting keypoints"):
            # Build a unique ID from the parent folder + filename
            category = vid_path.parent.name.replace(' ', '_')
            stem = vid_path.stem.replace(' ', '_')
            video_id = f"{category}___{stem}"
            out_file = OUTPUT_DIR / f"{video_id}.npz"
            
            if out_file.exists():
                total_extracted += 1
                continue  # Already processed
            
            try:
                kp = extractor.extract_video(vid_path)
                if kp is not None:
                    np.savez_compressed(out_file, keypoints=kp)
                    total_extracted += 1
                else:
                    total_failed += 1
            except Exception as e:
                total_failed += 1
                print(f"  ❌ {vid_path.name}: {e}")
        
        print(f"✅ Done with {zip_name}. Running total: {total_extracted} extracted, {total_failed} failed.")
        
    except Exception as e:
        print(f"❌ Failed to process {zip_name}: {e}")

extractor.close()

# Cleanup
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)

print(f"\n{'='*60}")
print(f"🎉 EXTRACTION COMPLETE!")
print(f"Total keypoint files: {total_extracted}")
print(f"Total failures: {total_failed}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"{'='*60}")

Fetching INCLUDE metadata from Zenodo...


Found 44 zip files to process.


✅ MediaPipe Holistic initialized (543 landmarks/frame)

[1/44] Processing Adjectives_3of8.zip


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773056238.419905      72 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


W0000 00:00:1773056238.616728      72 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773056238.618310      70 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773056238.618391      71 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773056238.621559      72 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773056238.641006      72 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773056238.656688      70 inference_feedback_manager.cc:114] Feedback manager 

...

✅ Done with Colours_1of2.zip. Running total: 2918 extracted, 0 failed.

[31/44] Processing Electronics_2of2.zip


Electronics_2of2.zip:   0%|          | 0.00/824M [00:00<?, ?iB/s]

Extracting zip...


Found 70 video files.


Extracting keypoints:   0%|          | 0/70 [00:00<?, ?it/s]

In [ ]:
# Cell 4: Verify output
from pathlib import Path
import numpy as np

output = Path('/kaggle/working/data/processed/include')
npz_files = list(output.glob('*.npz'))
print(f"Total .npz files: {len(npz_files)}")

if npz_files:
    sample = np.load(npz_files[0])
    print(f"Sample file: {npz_files[0].name}")
    print(f"Keypoints shape: {sample['keypoints'].shape}")
    print(f"  → {sample['keypoints'].shape[0]} frames × {sample['keypoints'].shape[1]} landmarks × {sample['keypoints'].shape[2]} coords")